# 🚀 KIRO2 - YOLO Eğitimi (Google Colab Pro+ / A100)

**Hedef:** 1,934 annotation ile soru detection modeli eğitmek  
**GPU:** A100 (40GB VRAM)  
**Süre:** ~1-2 saat  
**Sonuç:** %85-90 doğruluk

---

## ⚙️ 1. KURULUM VE KONTROLLER

In [ ]:
# GPU Kontrolü
!nvidia-smi

import torch
print(f"\n✅ PyTorch Version: {torch.__version__}")
print(f"✅ CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

In [ ]:
# Gerekli kütüphaneleri yükle
!pip install -q ultralytics tqdm

from ultralytics import YOLO
print("✅ Ultralytics kuruldu!")

---

## 📁 2. GOOGLE DRIVE BAĞLANTISI

In [ ]:
# Google Drive'ı bağla
from google.colab import drive
drive.mount('/content/drive')

print("\n✅ Google Drive bağlandı!")
print("📁 Drive içeriği:")
!ls -lh /content/drive/MyDrive/

---

## 📦 3. VERİSETİNİ YÜKLE

**Önemli:** `kiro2_annotations.zip` dosyasını Google Drive'ın ana dizinine yüklemiş olmalısın!

In [ ]:
import zipfile
import os

# ZIP dosyasının yolu (Google Drive'da)
zip_path = '/content/drive/MyDrive/kiro2_annotations.zip'

# Çıkartma klasörü
extract_path = '/content/kiro2_veriseti'

print("🗜️ ZIP dosyası çıkartılıyor...")
print(f"📂 Kaynak: {zip_path}")
print(f"📂 Hedef: {extract_path}\n")

# ZIP'i çıkart
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("\n✅ ZIP çıkartıldı!")
print("\n📊 İçerik:")
!ls -lh /content/kiro2_veriseti/

# JSON sayısını kontrol et
import glob
json_files = glob.glob('/content/kiro2_veriseti/**/*.json', recursive=True)
print(f"\n✅ Toplam JSON: {len(json_files)}")

---

## 🔄 4. LABELME → YOLO CONVERSION

In [ ]:
%%writefile labelme_to_yolo_converter.py
# LabelMe to YOLO Converter - Colab Version

import json
import os
from pathlib import Path
from typing import List, Dict, Tuple
import shutil
from tqdm import tqdm
import random

class LabelMeToYOLO:
    def __init__(self, class_names: List[str]):
        self.class_names = class_names
        self.class_to_id = {name: idx for idx, name in enumerate(class_names)}
    
    def convert_bbox(self, points: List[List[float]], img_width: int, img_height: int) -> Tuple[float, float, float, float]:
        x_coords = [p[0] for p in points]
        y_coords = [p[1] for p in points]
        
        x_min = min(x_coords)
        x_max = max(x_coords)
        y_min = min(y_coords)
        y_max = max(y_coords)
        
        x_center = (x_min + x_max) / 2.0 / img_width
        y_center = (y_min + y_max) / 2.0 / img_height
        width = (x_max - x_min) / img_width
        height = (y_max - y_min) / img_height
        
        return x_center, y_center, width, height
    
    def convert_annotation(self, json_path: Path, output_dir: Path, images_output_dir: Path) -> bool:
        try:
            with open(json_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            img_height = data.get('imageHeight')
            img_width = data.get('imageWidth')
            
            if not img_height or not img_width:
                return False
            
            label_filename = json_path.stem + '.txt'
            label_path = output_dir / label_filename
            
            yolo_lines = []
            
            for shape in data.get('shapes', []):
                label = shape.get('label')
                points = shape.get('points')
                shape_type = shape.get('shape_type')
                
                if shape_type != 'rectangle':
                    continue
                
                if label not in self.class_to_id:
                    continue
                
                class_id = self.class_to_id[label]
                
                x_center, y_center, width, height = self.convert_bbox(
                    points, img_width, img_height
                )
                
                yolo_line = f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}"
                yolo_lines.append(yolo_line)
            
            if yolo_lines:
                with open(label_path, 'w') as f:
                    f.write('\n'.join(yolo_lines))
                
                img_filename = data.get('imagePath')
                if img_filename:
                    img_path = json_path.parent / img_filename
                    
                    if not img_path.exists():
                        img_path = json_path.parent / (json_path.stem + '.png')
                    
                    if img_path.exists():
                        img_output_path = images_output_dir / img_path.name
                        shutil.copy2(img_path, img_output_path)
                        return True
                return False
            return False
                
        except Exception as e:
            return False
    
    def convert_dataset(
        self, 
        annotation_sources: List[Path],
        output_base_dir: Path,
        train_ratio: float = 0.8
    ):
        train_images_dir = output_base_dir / 'images' / 'train'
        val_images_dir = output_base_dir / 'images' / 'val'
        train_labels_dir = output_base_dir / 'labels' / 'train'
        val_labels_dir = output_base_dir / 'labels' / 'val'
        
        for dir_path in [train_images_dir, val_images_dir, train_labels_dir, val_labels_dir]:
            dir_path.mkdir(parents=True, exist_ok=True)
        
        all_json_files = []
        for source_dir in annotation_sources:
            if source_dir.exists():
                json_files = list(source_dir.rglob('*.json'))
                all_json_files.extend(json_files)
        
        print(f"📊 Toplam {len(all_json_files)} JSON dosyası bulundu")
        
        random.shuffle(all_json_files)
        
        split_idx = int(len(all_json_files) * train_ratio)
        train_files = all_json_files[:split_idx]
        val_files = all_json_files[split_idx:]
        
        print(f"✂️ Split: {len(train_files)} train, {len(val_files)} val")
        
        print("\n🔄 Train set dönüştürülüyor...")
        train_success = 0
        for json_path in tqdm(train_files, desc="Train"):
            if self.convert_annotation(json_path, train_labels_dir, train_images_dir):
                train_success += 1
        
        print("\n🔄 Val set dönüştürülüyor...")
        val_success = 0
        for json_path in tqdm(val_files, desc="Val"):
            if self.convert_annotation(json_path, val_labels_dir, val_images_dir):
                val_success += 1
        
        print(f"\n✅ Train: {train_success}/{len(train_files)} başarılı")
        print(f"✅ Val: {val_success}/{len(val_files)} başarılı")
        
        self.create_yaml_config(output_base_dir)
        
        print(f"\n🎉 Dataset hazır: {output_base_dir}")
        return train_success, val_success
    
    def create_yaml_config(self, output_base_dir: Path):
        yaml_content = f"""# KIRO2 Dataset Configuration
path: {output_base_dir.absolute()}
train: images/train
val: images/val

names:
"""
        for idx, name in enumerate(self.class_names):
            yaml_content += f"  {idx}: {name}\n"
        
        yaml_path = output_base_dir / 'dataset.yaml'
        with open(yaml_path, 'w', encoding='utf-8') as f:
            f.write(yaml_content)
        
        print(f"📝 Config dosyası: {yaml_path}")

# Main execution
if __name__ == '__main__':
    class_names = ['soru', 'cevaplar', 'konu', 'sayfa', 'test no']
    converter = LabelMeToYOLO(class_names)
    
    base_path = Path('/content/kiro2_veriseti')
    
    annotation_sources = [
        base_path / 'annotation' / 'images',
        base_path / 'zkitap' / 'screenshots'
    ]
    
    output_dir = Path('/content/yolo_dataset')
    
    print("🚀 KIRO2 - LabelMe → YOLO Converter")
    print("=" * 60)
    print(f"📂 Output: {output_dir}")
    print(f"🏷️ Classes: {class_names}")
    print("=" * 60)
    
    train_success, val_success = converter.convert_dataset(
        annotation_sources=annotation_sources,
        output_base_dir=output_dir,
        train_ratio=0.8
    )
    
    print("\n✅ Conversion tamamlandı!")

In [ ]:
# Conversion'ı çalıştır
!python labelme_to_yolo_converter.py

---

## 🎯 5. YOLO MODELİ EĞİTİMİ (A100 ile ~1-2 saat)

In [ ]:
from ultralytics import YOLO
import torch

print("🚀 KIRO2 - YOLO Model Eğitimi (A100)")
print("=" * 80)

# GPU kontrolü
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("⚠️ GPU bulunamadı!")

# Model yükle - A100 için büyük model kullanabiliriz!
model_size = 'm'  # A100'de medium model kullanabiliriz (n yerine)
model = YOLO(f'yolo11{model_size}.pt')

print(f"\n📦 Model: yolo11{model_size}.pt")
print("\n⚙️ Eğitim Parametreleri:")
print("   Dataset: /content/yolo_dataset/dataset.yaml")
print("   Epochs: 100")
print("   Batch Size: 32 (A100 sayesinde!)")
print("   Image Size: 640")
print("\n🎯 Eğitim başlatılıyor...")
print("=" * 80)

In [ ]:
# EĞİTİM BAŞLAT - A100 ile optimize edilmiş parametreler
results = model.train(
    data='/content/yolo_dataset/dataset.yaml',
    epochs=100,
    imgsz=640,
    batch=32,  # A100 ile büyük batch size
    device=0,
    
    # Optimization
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    
    # Augmentation
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=0.0,
    translate=0.1,
    scale=0.5,
    shear=0.0,
    perspective=0.0,
    flipud=0.0,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.0,
    
    # Validation
    val=True,
    save=True,
    save_period=10,
    
    # Logging
    project='runs/detect',
    name='kiro2_soru_detection',
    exist_ok=True,
    
    # Advanced
    patience=50,
    plots=True,
    verbose=True,
    
    # A100 için ek optimizasyonlar
    amp=True,  # Mixed precision training
    workers=8  # Çok thread kullan
)

print("\n" + "=" * 80)
print("✅ EĞİTİM TAMAMLANDI!")
print("=" * 80)

---

## 📊 6. SONUÇLARI İNCELE

In [ ]:
# Sonuçları göster
from IPython.display import Image, display
import matplotlib.pyplot as plt

print("📊 EĞİTİM SONUÇLARI")
print("=" * 80)

# mAP değerleri
print(f"\n📈 Metrikler:")
print(f"   mAP@50: {results.results_dict.get('metrics/mAP50(B)', 0):.4f}")
print(f"   mAP@50-95: {results.results_dict.get('metrics/mAP50-95(B)', 0):.4f}")
print(f"   Precision: {results.results_dict.get('metrics/precision(B)', 0):.4f}")
print(f"   Recall: {results.results_dict.get('metrics/recall(B)', 0):.4f}")

print(f"\n📁 Model Konumu:")
print(f"   Best: runs/detect/kiro2_soru_detection/weights/best.pt")
print(f"   Last: runs/detect/kiro2_soru_detection/weights/last.pt")

# Grafikleri göster
print("\n📊 Eğitim Grafikleri:")
display(Image(filename='runs/detect/kiro2_soru_detection/results.png'))
display(Image(filename='runs/detect/kiro2_soru_detection/confusion_matrix.png'))

---

## 🧪 7. MODELİ TEST ET

In [ ]:
# En iyi modeli yükle
best_model = YOLO('runs/detect/kiro2_soru_detection/weights/best.pt')

# Validation set'ten örnek test
import os
import random

val_images = os.listdir('/content/yolo_dataset/images/val')
test_image = random.choice(val_images)
test_image_path = f'/content/yolo_dataset/images/val/{test_image}'

print(f"🧪 Test görseli: {test_image}")

# Prediction
results = best_model.predict(
    source=test_image_path,
    save=True,
    conf=0.25
)

# Sonucu göster
print(f"\n✅ Tespit edilen: {len(results[0].boxes)} nesne")
for i, box in enumerate(results[0].boxes):
    cls = int(box.cls)
    conf = float(box.conf)
    class_name = ['soru', 'cevaplar', 'konu', 'sayfa', 'test no'][cls]
    print(f"   {i+1}. {class_name} (güven: {conf:.2f})")

# Görseli göster
display(Image(filename='runs/detect/predict/'+test_image))

---

## 💾 8. MODELİ KAYDET (Google Drive)

In [ ]:
# Best modeli Google Drive'a kopyala
import shutil

drive_save_path = '/content/drive/MyDrive/kiro2_yolo_model'
os.makedirs(drive_save_path, exist_ok=True)

# Model dosyasını kopyala
shutil.copy2(
    'runs/detect/kiro2_soru_detection/weights/best.pt',
    f'{drive_save_path}/kiro2_best.pt'
)

# Grafikleri kopyala
shutil.copy2(
    'runs/detect/kiro2_soru_detection/results.png',
    f'{drive_save_path}/results.png'
)

shutil.copy2(
    'runs/detect/kiro2_soru_detection/confusion_matrix.png',
    f'{drive_save_path}/confusion_matrix.png'
)

print("✅ Model Google Drive'a kaydedildi!")
print(f"📁 Konum: {drive_save_path}")
print("\n📦 Kaydedilen dosyalar:")
!ls -lh {drive_save_path}

---

## 📤 9. ONNX EXPORT (Production İçin)

In [ ]:
# ONNX formatına export et
print("📦 ONNX Export...")

onnx_path = best_model.export(format='onnx')

print(f"✅ ONNX export tamamlandı!")
print(f"📁 Path: {onnx_path}")

# ONNX'i de Google Drive'a kopyala
shutil.copy2(onnx_path, f'{drive_save_path}/kiro2_best.onnx')
print(f"\n✅ ONNX dosyası da Google Drive'a kaydedildi!")

---

## 🎉 TAMAMLANDI!

### ✅ Yapılanlar:
1. ✅ 1,934 annotation YOLO formatına çevrildi
2. ✅ YOLOv11 modeli A100'de eğitildi
3. ✅ Model test edildi
4. ✅ Google Drive'a kaydedildi
5. ✅ ONNX formatına export edildi

### 📁 Google Drive İçeriği:
```
MyDrive/kiro2_yolo_model/
├── kiro2_best.pt           ← PyTorch model
├── kiro2_best.onnx         ← ONNX model (production)
├── results.png             ← Eğitim grafikleri
└── confusion_matrix.png    ← Confusion matrix
```

### 🚀 Sonraki Adımlar:
1. `kiro2_best.pt` dosyasını bilgisayarına indir
2. Backend'e entegre et
3. OCR pipeline'ı kur
4. Production'a al!